In [19]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt
import aerosandbox as asb

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed
from Drag.Fuselage import Fuselage
from Drag.Bay import Bay
from Drag.LandingGear import LandingGear
from Aircraft.Aircraft import Aircraft
from global_parameters import Assumptions
from Requirements.FuelReq import FuelReq
from Requirements.LGReq import LGReq
from Requirements.MassReq import MassReq
from Requirements.MDReq import MDReq
from Requirements.EmpennageReq import EmpennageReq
from Requirements.Requirement import Requirement
from EmpennageSizing.TailFinder import TailFinder
from EmpennageSizing.CanardFinder import CanardFinder
from structural_analysis.Material import Material

# Loading the pre-computed planforms and the fuselage

In [20]:
with open("pickles/planform_pickle_official.pcl", "r+b") as f:
    planforms_recovered:list[tuple[Planform, str, bool]] = pickle.load(f)

assumptions = Assumptions()

In [21]:
# --- Import Onshape pull utilities ---
sys.path.append(os.path.abspath(os.getcwd()))
from onshape_pull import (
    fetch_variable_studio, fetch_measurement_features,
    evaluate_measurements, load_cached_masses, fetch_mass_properties,
    compute_cg_scenarios, lookup_var, lookup_meas,
    UPDATE_MASSES,
)

# --- Pull data from Onshape ---
variables = fetch_variable_studio()
meas_names = fetch_measurement_features()
measurements = evaluate_measurements(meas_names)
components = load_cached_masses() if not UPDATE_MASSES else fetch_mass_properties()
cg_data = compute_cg_scenarios(components)



  → Found 78 occurrences, fetching mass properties...
    ... processed 10/78 occurrences
    ... processed 20/78 occurrences
    ... processed 30/78 occurrences
    ... processed 40/78 occurrences
    ... processed 50/78 occurrences
    ... processed 60/78 occurrences
    ... processed 70/78 occurrences
  → Mass data cached to mass_cache.json


In [22]:
# --- Z offset (axle datum) ---
Front_Landing_Gear_Hinge_Z = lookup_meas(measurements, "Front_Landing_Gear_Hinge_Z")
Front_Strut_Height, _, _ = lookup_var(variables, "Front_Strut_Height")
Front_Gear_Extension_Max = lookup_meas(measurements, "Front_Gear_Extension_Max")
Front_Gear_Extension_Min = lookup_meas(measurements, "Front_Gear_Extension_Min")
z_offset = (Front_Landing_Gear_Hinge_Z + Front_Strut_Height
            + (Front_Gear_Extension_Max - Front_Gear_Extension_Min))

# --- Build drag components ---
engine_bay = Bay(
    surface_wetted=83744.32631 / 1e6,  # mm² → m² (hardcoded, not in Onshape)
    length=0.172,                       # 172 mm (hardcoded, not in Onshape)
    diameter=lookup_var(variables, "engine_diameter")[0],
)

Front_Gear_Unexposed = lookup_meas(measurements, "Front_Gear_Unexposed")
nose_gear = LandingGear(
    wheel_width=0.025,
    exposed_height=Front_Strut_Height - Front_Gear_Unexposed,
    wheel_diameter=lookup_var(variables, "Wheel_Diameter")[0],
    strut_width=lookup_var(variables, "Front_Strut_Diameter")[0],
)

Rear_Strut_Height, _, _ = lookup_var(variables, "Rear_Strut_Height")
Rear_Strut_height_2, _, _ = lookup_var(variables, "Rear_Strut_height_2")
main_gear = LandingGear(
    wheel_width=0.025,
    exposed_height=Rear_Strut_Height + Rear_Strut_height_2,
    wheel_diameter=lookup_var(variables, "Wheel_Diameter")[0],
    strut_width=lookup_var(variables, "Rear_Strut_Diameter")[0],
)

fus_len = lookup_var(variables, "FuselageLength")[0]

fuselage = Fuselage(
    surface_wetted=lookup_meas(measurements, "Wetted_Area"),
    length_total=fus_len,
    diameter_max=lookup_var(variables, "FuselageHeight")[0],
    upsweep=0.0,
    base_area=lookup_meas(measurements, "Base_Area"),
)

# --- X-position helpers ---
WingPortDistance = fus_len/2 - .025 # m #lookup_var(variables, "WingPortDistance")
WingPortWidth, _, _ = lookup_var(variables, "WingPortWidth")
CanardPortXLoc, _, _ = lookup_var(variables, "CanardPortXLoc")
CanardPortWidth, _, _ = lookup_var(variables, "CanardPortWidth")

print(variables)
print(WingPortDistance, WingPortWidth, CanardPortWidth, CanardPortWidth)

# --- Fixed parameters ---
fixed = Fixed(
    mass=cg_data["mass"],
    fuel_mass=cg_data["fuel_mass"],
    x_cg_min=cg_data["x_cg_min"],
    x_cg_max=cg_data["x_cg_max"],
    x_tail_cone=lookup_meas(measurements, "Tailcone_X"),
    z_cg=cg_data["z_cg_full"] + z_offset,
    z_tail_cone=-lookup_meas(measurements, "Z_TailCone") + z_offset,
    z_wing=lookup_meas(measurements, "Z_wing_LE_Abs") + z_offset,
    x_LE_canard=CanardPortXLoc + CanardPortWidth / 2,
    x_LE_wing=WingPortDistance + 0.115 - WingPortWidth / 2,
    x_LE_tail=lookup_meas(measurements, "X_LE_Tail"),
    x_nose_gear=lookup_meas(measurements, "x_nose_gear"),
    x_main_gear=lookup_meas(measurements, "x_main_gear"),
    y_main_gear=0.419,
    fuselage=fuselage,
    nose_gear=nose_gear,
    main_gear=main_gear,
    engine_bay=engine_bay,
)

[{'name': 'CANARD_VARIABLES', 'type': 'ANY', 'expression': '0', 'value': 0.0, 'unit': '', 'description': '================'}, {'name': 'CanardPortXLoc', 'type': 'LENGTH', 'expression': '250 mm', 'value': 250.0, 'unit': 'mm', 'description': ''}, {'name': 'CanardPortThickness', 'type': 'LENGTH', 'expression': '65 mm', 'value': 65.0, 'unit': 'mm', 'description': ''}, {'name': 'CanardPortWidth', 'type': 'LENGTH', 'expression': '150 mm', 'value': 150.0, 'unit': 'mm', 'description': ''}, {'name': 'CanardArea', 'type': 'ANY', 'expression': '0.03', 'value': 0.03, 'unit': '', 'description': 'm2'}, {'name': 'CanardTaperRatio', 'type': 'ANY', 'expression': '0.5', 'value': 0.5, 'unit': '', 'description': ''}, {'name': 'CanardAspectRatio', 'type': 'ANY', 'expression': '3', 'value': 3.0, 'unit': '', 'description': ''}, {'name': 'CanardSpan', 'type': 'ANY', 'expression': 'sqrt(#CanardAspectRatio*#CanardArea)', 'value': None, 'unit': 'sqrt(#CanardAspectRatio*#CanardArea)', 'description': 'm, Do not ch

In [23]:
with open("pickles/fixed_pickle.pcl", "wb") as f:
    pickle.dump(fixed, f)

In [24]:
with open("pickles/fixed_pickle.pcl", "rb") as f:
    fixed:Fixed = pickle.load(f)

In [25]:
for component in fixed.drag_components(False):
    component.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
    component.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
    component.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
for component in fixed.drag_components(True):
    component.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)

In [26]:
print(fixed.fuselage.characteristic_length, fixed.x_cg_min, fixed.x_cg_min, fixed.x_LE_wing)

2.7 1.0317893591765872 1.0317893591765872 1.3150000000000002


# Creating full Aircraft objects

In [ ]:
material_skin = Material(assumptions.cfrp_density, elastic_modulus=assumptions.cfrp_Young_modulus, 
                         poisson_ratio=assumptions.cfrp_poisson, shear_modulus=assumptions.cfrp_Young_modulus / 2 / (1 + assumptions.cfrp_poisson),
                         yield_strength=assumptions.cfrp_yield_strength, fracture_strength=assumptions.cfrp_yield_strength)

In [ ]:
aircraft:list[Aircraft] = list()

for i, planform_recovered in enumerate(planforms_recovered):
    main_wing = planform_recovered[0]
    planform_type = planform_recovered[1]

    ef = TailFinder(fixed, material=material_skin, core_density=assumptions.foam_denisty, thicknesses=assumptions.allowable_thicknesses, safety_factor=assumptions.structural_safety_factor, AR_h=5.) if (planform_type == "tail") else CanardFinder(fixed, material=material_skin, core_density=assumptions.foam_denisty, thicknesses=assumptions.allowable_thicknesses, safety_factor=assumptions.structural_safety_factor, AR_c=max(5., main_wing.aspect_ratio/1.5))
    
    emp = ef.find_planforms(main_wing, print_=i==28)

    for e in emp:
        e.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
        e.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
        e.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
        e.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)
        e.mass_cache = 1
        e.x_cg_cache = .1

    aircraft_planforms = [main_wing] + emp #TODO add the empenage
    aircraft.append(Aircraft(
        fixed=fixed, #TODO: add the fuselage from CAD
        planforms=aircraft_planforms 
    ))

0.3415475767479746
Stresses 661665.8918851808, 2513038.8733308823, 0.0004
0.3415475767479748
Stresses 2002234.8966882539, 26203305.478651352, 0.0004
0.4683908937165085
Stresses 838464.9201269819, 3168851.3036171338, 0.0004
0.4683908937165087
Stresses 2164819.624761847, 23159204.127692144, 0.0004
0.45251679668196243
Stresses 817065.5277433044, 3100638.594728868, 0.0004
0.45251679668196243
Stresses 2146457.7440938274, 23545824.817325212, 0.0004
0.454262125814171
Stresses 819427.4329657367, 3150827.015199075, 0.0004
0.45426212581417097
Stresses 2148500.2537891585, 23824312.764663767, 0.0004
0.4540673909186716
Stresses 819164.0162995977, 3101174.4539098875, 0.0004
0.45406739091867143
Stresses 2148272.655961863, 23460480.545102425, 0.0004
0.4540782368899806
Stresses 819178.6883195532, 3107134.147655107, 0.0004
0.45407823688998045
Stresses 2148285.3342141206, 23504883.386266135, 0.0004
0.3415475767479746
Stresses 661665.8918851808, 2513038.8733308823, 0.0004
0.3412436975038259
Stresses 33233

In [ ]:
s_ratios = [ac.planforms[1].wing_area / ac.planforms[0].wing_area for ac in aircraft]
print(s_ratios)
print(min(s_ratios), np.average(s_ratios), max(s_ratios))
idx_bad = np.argmax(s_ratios)
ac_bad:Aircraft = aircraft[idx_bad],
ac_bad= ac_bad[0]
print(ac_bad.planforms[0].sweep_quarter_rad, ac_bad.planforms[0].aspect_ratio, ac_bad.planforms[0].cm_quarter_chord, ac_bad.planforms[0].thickness_to_chord, planforms_recovered[idx_bad][1])

[0.11531271854031137, 0.05102398098356617, 0.11815202928566051, 0.16334790549096415, 0.38753056722229406, 0.423313399354861, 0.2182470338521758, 0.2175752948674255, 0.11577623132789029, 0.050222830395130244, 0.11871991204771092, 0.16415521541475633, 0.38680347617167793, 0.42236861016588945, 0.21747257124264746, 0.2167004669491315, 0.11625939496047874, 0.0493872313808923, 0.11931210657588459, 0.1649972843890114, 0.386044903068544, 0.42138347957266564, 0.21666450693134964, 0.2157883481827833, 0.21460784197517235, 0.12263071049465694, 0.21460784197517235, 0.2703716975154594, 0.3920451421038749, 0.3647021256131258, 0.27035053812637205, 0.23660616600480947, 0.21072645640339516, 0.11609321324921545, 0.21072645640339516, 0.26386391771349726, 0.39882756209375275, 0.37219477278815155, 0.27743992343411306, 0.24374131887347616, 0.20904691116935847, 0.11326823277528238, 0.20904691116935847, 0.26105191570699754, 0.4017527799874088, 0.37543789428149943, 0.2804965759646967, 0.24683048515259184, 0.318

# Checking if reuirements are met

In [ ]:
requirements:list[Requirement] = [
    MassReq(50.),
    MDReq(),
    FuelReq(),
    LGReq(),
    EmpennageReq(),
]

requirement_labels = [
    "MTOM",
    "Matching Diagram",
    "Fuel",
    "Landing Gear",
    "Empennage Requirement"
]

In [ ]:
for ac in aircraft:
    failed_reqs = list()
    for requirement, label in zip(requirements, requirement_labels):
        if not requirement.assess(ac):
            failed_reqs.append(label)

    if len(failed_reqs):
        print(f"ac mass: {ac.total_mass()}, {ac.planforms[0].oswald}")
        print(f"MainWing: AR={ac.planforms[0].aspect_ratio}, tc={ac.planforms[0].thickness_to_chord}, sweep={np.rad2deg(ac.planforms[0].sweep_quarter_rad)} deg, cmac={ac.planforms[0].cm_quarter_chord}")
        print(f"Failed: {failed_reqs}")
        print()

Fuel available: 11.000000001437998 kg
Fuel required: 7.009019028012672 kg
Difference: 3.9909809734253265 kg
all constraints satisfied
Fuel available: 11.000000001437998 kg
Fuel required: 7.022578454835737 kg
Difference: 3.977421546602261 kg
all constraints satisfied
Fuel available: 11.000000001437998 kg
Fuel required: 7.172962019260426 kg
Difference: 3.827037982177572 kg
all constraints satisfied
Fuel available: 11.000000001437998 kg
Fuel required: 7.110699205206417 kg
Difference: 3.8893007962315815 kg
all constraints satisfied
Fuel available: 11.000000001437998 kg
Fuel required: 6.546244112929252 kg
Difference: 4.453755888508746 kg
all constraints satisfied
ac mass: 36.17975845823961, 0.7845238526961408
MainWing: AR=5.0, tc=0.06, sweep=14.999999999999998 deg, cmac=-0.15
Failed: ['Empennage Requirement']

Fuel available: 11.000000001437998 kg
Fuel required: 6.897015420418916 kg
Difference: 4.102984581019082 kg
all constraints satisfied
ac mass: 36.17975845823961, 0.7845238526961408
Mai